In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.sparse import coo_matrix
from tqdm.notebook import tqdm

In [2]:
# Importing the full lego matrix
components = pd.read_csv('data/components.tsv', sep='\t', index_col=0)
objects = pd.read_csv('data/objects.tsv', sep='\t', index_col=0)
count_sparse = pd.read_csv('data/count_sparse.zip', sep='\t', compression='zip')

In [3]:
# Grouping by shape (part_id)
agg_dict = {'name':'first', 'category':'first', 'material':'first', 'abundance':'sum', 'occurrence':'sum'}
shape_comp = components.groupby('part_id', as_index=False).agg(agg_dict)
shape_comp = shape_comp.sort_values('abundance', ascending=False)
shape_comp.index = np.arange(len(shape_comp))
shape_comp['sp_index'] = shape_comp.index

In [5]:
# Adding the new sparse indexing (after collapse) to the sparse matrix
count_sparse['part_id'] = count_sparse['component_id'].map(components['part_id'])
part_to_sparse = shape_comp.groupby('part_id').agg({'sp_index':'first'})
count_sparse['shape_id'] = count_sparse['part_id'].map(part_to_sparse['sp_index'])
count_sparse

,object_id,component_id,count,occurrence,part_id,shape_id
0,0,9046,1,1,21459,449
1,0,76560,1,1,970c22pr1604,4746
2,0,41582,1,1,40925pat0001,4226
3,0,81068,1,1,973c22h03pr4515,4771
4,0,36874,1,1,3626cpr9975,1936
...,...,...,...,...,...,...
1240229,33529,17958,4,1,3001a,78
1240230,33529,64856,2,1,7049bc01,706
1240231,33529,18269,1,1,3003a,101
1240232,33529,18461,1,1,3004,2


In [6]:
# Iterating over object and build the new sparse matrix by collapsing 
# counts at given part id
new_sparse_data = []
progress = tqdm(total=len(objects))
for obj_index in objects.index:
    obj_data = count_sparse[count_sparse['object_id'] == obj_index]
    obj_data = obj_data.groupby('part_id', as_index=False).agg({'shape_id':'first', 'object_id':'first', 'count':'sum'})
    new_sparse_data.append(np.array(obj_data[['shape_id', 'object_id', 'count']]))
    progress.update(1)
new_sparse_mat = pd.concat([pd.DataFrame(data = d, columns=['component_id', 'object_id', 'count']) for d in new_sparse_data])

  0%|          | 0/33530 [00:00<?, ?it/s]

In [7]:
# Re-coputing occurrences
new_sparse_mat['occurrence'] = 1
occ_map = new_sparse_mat.groupby('component_id').agg({'occurrence':'sum'})['occurrence']
shape_comp['occurrence'] = shape_comp.index.map(occ_map)

In [8]:
# Expoting

shape_comp = shape_comp.drop(['sp_index'], axis=1)
shape_comp.index.name = 'sparse_id'
shape_comp.to_csv('../legos_shape/data/components.tsv', sep='\t')

new_sparse_mat.drop(['occurrence'], axis=1).to_csv('../legos_shape/data/count_sparse.zip', sep='\t', index=None)